# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Schema URL: https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset name:", metadata.get('name'))
print("Description:", metadata.get('description'))
print("Authors (@id):", [author['@id'] for author in metadata.get('author', [])])
print("Record sets (@id):", metadata.get('recordSet', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset may contain multiple record sets (tables) described by their `@id`. We'll list all available record sets and, for each, print out the field and column IDs.

In [ ]:
# List all available record sets and fields
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"  - Record set @id: {rs['@id']} ({rs.get('name')})")
    fields = rs.get('fields', [])
    print("    Fields:")
    for field in fields:
        print(f"      * {field['@id']} ({field.get('name')})")
    columns = rs.get('columns', [])
    print("    Columns:")
    for col in columns:
        print(f"      * {col['@id']} ({col.get('name')})")


## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.
Each table is referenced by its record set `@id`. We'll extract the first record set for demonstration.

In [ ]:
# Extract data from each record set
dataframes = {}
available_record_set_ids = [rs['@id'] for rs in record_sets]
# For demonstration, use the first record set
primary_record_set_id = available_record_set_ids[0] if available_record_set_ids else None
print("Loading records for:", primary_record_set_id)

for record_set_id in available_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if primary_record_set_id:
    print("Columns:", dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# For demonstration, let's select a numeric field from the first record set
numeric_fields = []
for field in record_sets[0].get('fields', []):
    if field.get('dataType', '').lower() in ['integer', 'float', 'number']:
        numeric_fields.append(field['@id'])
# Use the first numeric field found
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = dataframes[primary_record_set_id].select_dtypes(include=['number']).columns[0]

# Define a filter threshold
threshold = 10
df = dataframes[primary_record_set_id]
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally group by a categorical field
    group_fields = [field['@id'] for field in record_sets[0].get('fields', []) if field.get('dataType', '').lower() == 'text']
    group_field = group_fields[0] if group_fields else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found in the current record set.")

## 5. Visualization
Visualize data distributions or relationships between fields. For example, plot the distribution of a numeric field, or relationships between two columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution if available
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If there is a categorical field, plot boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and process a FAIR^2 dataset using the `mlcroissant` library. The sample steps included reviewing metadata, extracting tables by `@id`, filtering, normalizing, grouping records, and visualizing distributions. For further analysis, refer to the detailed schema and field descriptions using their `@id` fields for consistent referencing.